## **Figure-S9**

In [1]:
import pandas as pd
import polars as pl
from collections import defaultdict
import sys
import numpy as np

import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from pyprophet.stats import pi0est, stat_metrics, final_err_table, error_statistics, qvalue
from scipy.stats import pearsonr

sys.path.append("../../scripts")

from figure_utils import *

In [2]:
print("Pandas Version", pd.__version__)
print("Polars Version", pl.__version__)
print("numpy Version", np.__version__)
print("matplotlib version", mpl.__version__)
print("seaborn version", sns.__version__)
print("Python Version", sys.version)

Pandas Version 2.2.3
Polars Version 1.28.1
numpy Version 2.2.2
matplotlib version 3.10.0
seaborn version 0.13.2
Python Version 3.12.4 (main, Jun  7 2024, 23:47:47) [GCC 13.3.0]


In [3]:
names = dict(exp='Experimental',
             silico='In-Silico',
             bruker='timsTOF PanHuman',
             panhuman='PanHuman, 2014')

condition = dict(orig='Initial', refined='Reconstructed', tl='Transfer-Learn', onlyFilter='Only-Filter')

## **Load Data**

#### **Load Identifications (For Jaccard Index)**

In [4]:
osw = { n:defaultdict(dict) for n in names.values() }

# Bruker data
osw[names['bruker']][condition['orig']]['1'] = getPrecursorSet("../../results/K562-Bruker-Library-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.parquet")
osw[names['bruker']][condition['orig']]['2'] = getPrecursorSet("../../results/K562-Bruker-Library-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.parquet")
osw[names['bruker']][condition['orig']]['3'] = getPrecursorSet("../../results/K562-Bruker-Library-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.parquet")

osw[names['bruker']][condition['refined']]['1'] = getPrecursorSet("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.parquet")
osw[names['bruker']][condition['refined']]['2'] = getPrecursorSet("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.parquet")
osw[names['bruker']][condition['refined']]['3'] = getPrecursorSet("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.parquet")

osw[names['bruker']][condition['tl']]['1'] = getPrecursorSet_oswpq("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/osw_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.oswpq")
osw[names['bruker']][condition['tl']]['2'] = getPrecursorSet_oswpq("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/osw_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.oswpq")
osw[names['bruker']][condition['tl']]['3'] = getPrecursorSet_oswpq("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/osw_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.oswpq")

osw[names['bruker']][condition['onlyFilter']]['1'] = getPrecursorSet("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/osw_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.parquet")
osw[names['bruker']][condition['onlyFilter']]['2'] = getPrecursorSet("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/osw_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.parquet")
osw[names['bruker']][condition['onlyFilter']]['3'] = getPrecursorSet("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/osw_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.parquet")

# Panhuman data
osw[names['panhuman']][condition['orig']]['1'] = getPrecursorSet_oswpq("../../results/K562-PanHuman-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_SVM/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.oswpq/")
osw[names['panhuman']][condition['orig']]['2'] = getPrecursorSet_oswpq("../../results/K562-PanHuman-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_SVM/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.oswpq/")
osw[names['panhuman']][condition['orig']]['3'] = getPrecursorSet_oswpq("../../results/K562-PanHuman-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_SVM/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.oswpq/")

osw[names['panhuman']][condition['refined']]['1'] = getPrecursorSet_oswpq("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_SVM/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.oswpq/")
osw[names['panhuman']][condition['refined']]['2'] = getPrecursorSet_oswpq("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_SVM/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.oswpq/")
osw[names['panhuman']][condition['refined']]['3'] = getPrecursorSet_oswpq("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_SVM/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.oswpq/")

osw[names['panhuman']][condition['tl'] ]['1'] = getPrecursorSet_oswpq("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/osw_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_SVM/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.oswpq/")
osw[names['panhuman']][condition['tl'] ]['2'] = getPrecursorSet_oswpq("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/osw_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_SVM/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.oswpq/")
osw[names['panhuman']][condition['tl'] ]['3'] = getPrecursorSet_oswpq("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/osw_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_SVM/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.oswpq/")

osw[names['panhuman']][condition['onlyFilter'] ]['1'] = getPrecursorSet_oswpq("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/osw_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_SVM/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.oswpq/")
osw[names['panhuman']][condition['onlyFilter'] ]['2'] = getPrecursorSet_oswpq("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/osw_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_SVM/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.oswpq/")
osw[names['panhuman']][condition['onlyFilter'] ]['3'] = getPrecursorSet_oswpq("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/osw_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_SVM/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.oswpq/")

# Experimental data
osw[names['exp']][condition['orig']]['1'] = getPrecursorSet("../../results/K562-Exp-Lib-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.parquet")
osw[names['exp']][condition['orig']]['2'] = getPrecursorSet("../../results/K562-Exp-Lib-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.parquet")
osw[names['exp']][condition['orig']]['3'] = getPrecursorSet("../../results/K562-Exp-Lib-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.parquet")

osw[names['exp']][condition['refined']]['1'] = getPrecursorSet("../../results/K562-Exp-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.parquet")
osw[names['exp']][condition['refined']]['2'] = getPrecursorSet("../../results/K562-Exp-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.parquet")
osw[names['exp']][condition['refined']]['3'] = getPrecursorSet("../../results/K562-Exp-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.parquet")

osw[names['exp']][condition['tl']]['1'] = getPrecursorSet_oswpq("../../results/K562-Exp-Lib-Refined-GPF-Analysis/osw_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.oswpq")
osw[names['exp']][condition['tl']]['2'] = getPrecursorSet_oswpq("../../results/K562-Exp-Lib-Refined-GPF-Analysis/osw_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.oswpq")
osw[names['exp']][condition['tl']]['3'] = getPrecursorSet_oswpq("../../results/K562-Exp-Lib-Refined-GPF-Analysis/osw_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.oswpq")

osw[names['exp']][condition['onlyFilter']]['1'] = getPrecursorSet("../../results/K562-Exp-Lib-Refined-GPF-Analysis/osw_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.parquet")
osw[names['exp']][condition['onlyFilter']]['2'] = getPrecursorSet("../../results/K562-Exp-Lib-Refined-GPF-Analysis/osw_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.parquet")
osw[names['exp']][condition['onlyFilter']]['3'] = getPrecursorSet("../../results/K562-Exp-Lib-Refined-GPF-Analysis/osw_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.parquet")


# Silico data
osw[names['silico']][condition['orig']]['1'] = getPrecursorSet_oswpq("../../results/K562-PeptDeep-NoMods-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.oswpq")
osw[names['silico']][condition['orig']]['2'] = getPrecursorSet_oswpq("../../results/K562-PeptDeep-NoMods-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.oswpq")
osw[names['silico']][condition['orig']]['3'] = getPrecursorSet_oswpq("../../results/K562-PeptDeep-NoMods-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.oswpq")

osw[names['silico']][condition['refined']]['1'] = getPrecursorSet("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.parquet")
osw[names['silico']][condition['refined']]['2'] = getPrecursorSet("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.parquet")
osw[names['silico']][condition['refined']]['3'] = getPrecursorSet("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.parquet")

osw[names['silico']][condition['tl'] ]['1'] = getPrecursorSet_oswpq("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/osw_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.oswpq")
osw[names['silico']][condition['tl'] ]['2'] = getPrecursorSet_oswpq("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/osw_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.oswpq")
osw[names['silico']][condition['tl'] ]['3'] = getPrecursorSet_oswpq("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/osw_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.oswpq")

osw[names['silico']][condition['onlyFilter']]['1'] = getPrecursorSet("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/osw_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.parquet")
osw[names['silico']][condition['onlyFilter']]['2'] = getPrecursorSet("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/osw_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.parquet")
osw[names['silico']][condition['onlyFilter']]['3'] = getPrecursorSet("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/osw_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.parquet")

../../results/K562-Bruker-Library-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.parquet
../../results/K562-Bruker-Library-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.parquet
../../results/K562-Bruker-Library-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.parquet
../../results/K562-Bruker-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.parquet
../../results/K562-Bruker-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Sl

In [5]:
diann = { n:defaultdict(dict) for n in names.values() }

# Bruker data
diann[names['bruker']][condition['orig']]['1'] = getPrecursorSetDiann("../../results/K562-Bruker-Library-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv")
diann[names['bruker']][condition['orig']]['2'] = getPrecursorSetDiann("../../results/K562-Bruker-Library-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/report.tsv")
diann[names['bruker']][condition['orig']]['3'] = getPrecursorSetDiann("../../results/K562-Bruker-Library-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/report.tsv")

diann[names['bruker']][condition['refined']]['1'] = getPrecursorSetDiann("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv")
diann[names['bruker']][condition['refined']]['2'] = getPrecursorSetDiann("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/report.tsv")
diann[names['bruker']][condition['refined']]['3'] = getPrecursorSetDiann("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/report.tsv")

diann[names['bruker']][condition['tl']]['1'] = getPrecursorSetDiann("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/diann_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv")
diann[names['bruker']][condition['tl']]['2'] = getPrecursorSetDiann("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/diann_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/report.tsv")
diann[names['bruker']][condition['tl']]['3'] = getPrecursorSetDiann("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/diann_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/report.tsv")

diann[names['bruker']][condition['onlyFilter']]['1'] = getPrecursorSetDiann("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/diann_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv")
diann[names['bruker']][condition['onlyFilter']]['2'] = getPrecursorSetDiann("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/diann_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/report.tsv")
diann[names['bruker']][condition['onlyFilter']]['3'] = getPrecursorSetDiann("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/diann_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/report.tsv")


# Silico data
diann[names['silico']][condition['orig']]['1'] = getPrecursorSetDiann("../../results/K562-PeptDeep-NoMods-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv")
diann[names['silico']][condition['orig']]['2'] = getPrecursorSetDiann("../../results/K562-PeptDeep-NoMods-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/report.tsv")
diann[names['silico']][condition['orig']]['3'] = getPrecursorSetDiann("../../results/K562-PeptDeep-NoMods-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/report.tsv")

diann[names['silico']][condition['refined']]['1'] = getPrecursorSetDiann("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv")
diann[names['silico']][condition['refined']]['2'] = getPrecursorSetDiann("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/report.tsv")
diann[names['silico']][condition['refined']]['3'] = getPrecursorSetDiann("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/report.tsv")

diann[names['silico']][condition['tl']]['1'] = getPrecursorSetDiann("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/diann_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv")
diann[names['silico']][condition['tl']]['2'] = getPrecursorSetDiann("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/diann_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/report.tsv")
diann[names['silico']][condition['tl']]['3'] = getPrecursorSetDiann("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/diann_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/report.tsv")

diann[names['silico']][condition['onlyFilter']]['1'] = getPrecursorSetDiann("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/diann_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv")
diann[names['silico']][condition['onlyFilter']]['2'] = getPrecursorSetDiann("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/diann_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/report.tsv")
diann[names['silico']][condition['onlyFilter']]['3'] = getPrecursorSetDiann("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/diann_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/report.tsv")


# exp data
diann[names['exp']][condition['orig']]['1'] = getPrecursorSetDiann("../../results/K562-Exp-Lib-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv")
diann[names['exp']][condition['orig']]['2'] = getPrecursorSetDiann("../../results/K562-Exp-Lib-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/report.tsv")
diann[names['exp']][condition['orig']]['3'] = getPrecursorSetDiann("../../results/K562-Exp-Lib-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/report.tsv")

diann[names['exp']][condition['refined']]['1'] = getPrecursorSetDiann("../../results/K562-Exp-Lib-Refined-GPF-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv", infer_schema_length=1000)
diann[names['exp']][condition['refined']]['2'] = getPrecursorSetDiann("../../results/K562-Exp-Lib-Refined-GPF-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/report.tsv", infer_schema_length=1000)
diann[names['exp']][condition['refined']]['3'] = getPrecursorSetDiann("../../results/K562-Exp-Lib-Refined-GPF-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/report.tsv", infer_schema_length=1000)

diann[names['exp']][condition['tl']]['1'] = getPrecursorSetDiann("../../results/K562-Exp-Lib-Refined-GPF-Analysis/diann_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv", infer_schema_length=1000)
diann[names['exp']][condition['tl']]['2'] = getPrecursorSetDiann("../../results/K562-Exp-Lib-Refined-GPF-Analysis/diann_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/report.tsv", infer_schema_length=1000)
diann[names['exp']][condition['tl']]['3'] = getPrecursorSetDiann("../../results/K562-Exp-Lib-Refined-GPF-Analysis/diann_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/report.tsv", infer_schema_length=1000)

diann[names['exp']][condition['onlyFilter']]['1'] = getPrecursorSetDiann("../../results/K562-Exp-Lib-Refined-GPF-Analysis/diann_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv", infer_schema_length=1000)
diann[names['exp']][condition['onlyFilter']]['2'] = getPrecursorSetDiann("../../results/K562-Exp-Lib-Refined-GPF-Analysis/diann_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/report.tsv", infer_schema_length=1000)
diann[names['exp']][condition['onlyFilter']]['3'] = getPrecursorSetDiann("../../results/K562-Exp-Lib-Refined-GPF-Analysis/diann_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/report.tsv", infer_schema_length=1000)


# Panhuman data
diann[names['panhuman']][condition['orig']]['1'] = getPrecursorSetDiann("../../results/K562-PanHuman-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv")
diann[names['panhuman']][condition['orig']]['2'] = getPrecursorSetDiann("../../results/K562-PanHuman-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/report.tsv")
diann[names['panhuman']][condition['orig']]['3'] = getPrecursorSetDiann("../../results/K562-PanHuman-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/report.tsv")

diann[names['panhuman']][condition['refined']]['1'] = getPrecursorSetDiann("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv")
diann[names['panhuman']][condition['refined']]['2'] = getPrecursorSetDiann("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/report.tsv")
diann[names['panhuman']][condition['refined']]['3'] = getPrecursorSetDiann("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/report.tsv")

diann[names['panhuman']][condition['tl']]['1'] = getPrecursorSetDiann("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/diann_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv")
diann[names['panhuman']][condition['tl']]['2'] = getPrecursorSetDiann("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/diann_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/report.tsv")
diann[names['panhuman']][condition['tl']]['3'] = getPrecursorSetDiann("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/diann_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/report.tsv")

diann[names['panhuman']][condition['onlyFilter']]['1'] = getPrecursorSetDiann("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/diann_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv")
diann[names['panhuman']][condition['onlyFilter']]['2'] = getPrecursorSetDiann("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/diann_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/report.tsv")
diann[names['panhuman']][condition['onlyFilter']]['3'] = getPrecursorSetDiann("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/diann_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/report.tsv")

../../results/K562-Bruker-Library-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv
../../results/K562-Bruker-Library-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/report.tsv
../../results/K562-Bruker-Library-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/report.tsv
../../results/K562-Bruker-Lib-Refined-GPF-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv
../../results/K562-Bruker-Lib-Refined-GPF-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/report.tsv
../../results/K562-Bruker-Lib-Refined-GPF-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/report.tsv
../../results/K562-Bruker-Lib-Refined-GPF-Analysis/diann_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv
../../results/K562-Bruker-Lib-Refined-GPF-Analysis/diann_tl/Rost_DIApy3_SP2u

### **Compute pi0 (Based on OSW)**

In [6]:
def compute_pi0_helper(targets, decoys):
    stats = error_statistics(targets, decoys, False, False, np.arange(0.1, 0.5, 0.1), pi0_method='bootstrap')
    final_stats = final_err_table(stats[0])
    
    return (stats[1]['pi0'], compute_area(final_stats))
    #return stats[1]['pi0'], final_stats


def compute_pi0(fPath):
    
    print(fPath)
    targets = pl.scan_parquet(fPath)
    targets = (targets.filter((pl.col('SCORE_MS2.RANK') == 1) & 
                              (pl.col('DECOY') == 0))
           .select('SCORE_MS2.SCORE')
           .collect()
           .to_series()
           .to_numpy())

    decoys = pl.scan_parquet(fPath)
    decoys = (decoys.filter((pl.col('SCORE_MS2.RANK') == 1) & 
                              (pl.col('DECOY') == 1))
           .select('SCORE_MS2.SCORE')
           .collect()
           .to_series()
           .to_numpy())

    return compute_pi0_helper(targets, decoys)

def compute_area(stat):
    y = np.concatenate([[0], stat['svalue'], [0.99]])
    x = np.concatenate([[0], stat['fdr'], [0.99]])
    return np.trapezoid(y, x)


def compute_pi0_oswpq(fPath):
    print(fPath)
    targets = pl.scan_parquet(fPath + '/precursors_features.parquet')
    targets = (targets.filter((pl.col('SCORE_MS2_PEAK_GROUP_RANK') == 1) & 
                              (pl.col('PRECURSOR_DECOY') == 0))
           .select('SCORE_MS2_SCORE')
           .collect()
           .to_series()
           .to_numpy())

    decoys = pl.scan_parquet(fPath + '/precursors_features.parquet')
    decoys = (decoys.filter((pl.col('SCORE_MS2_PEAK_GROUP_RANK') == 1) & 
                              (pl.col('PRECURSOR_DECOY') == 1))
           .select('SCORE_MS2_SCORE')
           .collect()
           .to_series()
           .to_numpy())

    return compute_pi0_helper(targets, decoys)

def get_num_features(fPath):
    print(fPath)
    parquet = pl.scan_parquet(fPath)
    return (parquet.filter((pl.col('SCORE_MS2.RANK') == 1) & 
                              (pl.col('DECOY') == 0))
           .count()
           .select('PRECURSOR_ID')
           .collect()
           .to_series() 
           .to_list())[0] 

def get_num_features_oswpq(fPath):
    print(fPath)
    parquet = pl.scan_parquet(fPath + '/precursors_features.parquet')
    
    return (parquet.filter((pl.col('SCORE_MS2_PEAK_GROUP_RANK') == 1) & 
                              (pl.col('PRECURSOR_DECOY') == 0))
           .select('PRECURSOR_ID') 
           .count()
           .collect()
           .to_series()
           .to_list())[0]
    


In [7]:
pi0s = { n:defaultdict(dict) for n in names.values() }

# Bruker data

pi0s[names['bruker']][condition['orig']]['1'] = compute_pi0("../../results/K562-Bruker-Library-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.parquet")
pi0s[names['bruker']][condition['orig']]['2'] = compute_pi0("../../results/K562-Bruker-Library-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.parquet")
pi0s[names['bruker']][condition['orig']]['3'] = compute_pi0("../../results/K562-Bruker-Library-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.parquet")

pi0s[names['bruker']][condition['refined']]['1'] = compute_pi0("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.parquet")
pi0s[names['bruker']][condition['refined']]['2'] = compute_pi0("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.parquet")
pi0s[names['bruker']][condition['refined']]['3'] = compute_pi0("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.parquet")

pi0s[names['bruker']][condition['tl']]['1'] = compute_pi0_oswpq("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/osw_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.oswpq")
pi0s[names['bruker']][condition['tl']]['2'] = compute_pi0_oswpq("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/osw_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.oswpq")
pi0s[names['bruker']][condition['tl']]['3'] = compute_pi0_oswpq("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/osw_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.oswpq")

pi0s[names['bruker']][condition['onlyFilter']]['1'] = compute_pi0("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/osw_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.parquet")
pi0s[names['bruker']][condition['onlyFilter']]['2'] = compute_pi0("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/osw_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.parquet")
pi0s[names['bruker']][condition['onlyFilter']]['3'] = compute_pi0("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/osw_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.parquet")

# Panhuman data
pi0s[names['panhuman']][condition['orig']]['1'] = compute_pi0_oswpq("../../results/K562-PanHuman-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_SVM/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.oswpq/")
pi0s[names['panhuman']][condition['orig']]['2'] = compute_pi0_oswpq("../../results/K562-PanHuman-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_SVM/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.oswpq/")
pi0s[names['panhuman']][condition['orig']]['3'] = compute_pi0_oswpq("../../results/K562-PanHuman-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_SVM/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.oswpq/")

pi0s[names['panhuman']][condition['refined']]['1'] = compute_pi0_oswpq("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_SVM/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.oswpq/")
pi0s[names['panhuman']][condition['refined']]['2'] = compute_pi0_oswpq("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_SVM/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.oswpq/")
pi0s[names['panhuman']][condition['refined']]['3'] = compute_pi0_oswpq("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_SVM/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.oswpq/")

pi0s[names['panhuman']][condition['tl'] ]['1'] = compute_pi0_oswpq("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/osw_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_SVM/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.oswpq/")
pi0s[names['panhuman']][condition['tl'] ]['2'] = compute_pi0_oswpq("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/osw_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_SVM/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.oswpq/")
pi0s[names['panhuman']][condition['tl'] ]['2'] = compute_pi0_oswpq("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/osw_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_SVM/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.oswpq/")

pi0s[names['panhuman']][condition['onlyFilter'] ]['1'] = compute_pi0_oswpq("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/osw_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_SVM/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.oswpq/")
pi0s[names['panhuman']][condition['onlyFilter'] ]['2'] = compute_pi0_oswpq("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/osw_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_SVM/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.oswpq/")
pi0s[names['panhuman']][condition['onlyFilter'] ]['2'] = compute_pi0_oswpq("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/osw_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_SVM/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.oswpq/")

# Experimental data
pi0s[names['exp']][condition['orig']]['1'] = compute_pi0("../../results/K562-Exp-Lib-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.parquet")
pi0s[names['exp']][condition['orig']]['2'] = compute_pi0("../../results/K562-Exp-Lib-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.parquet")
pi0s[names['exp']][condition['orig']]['3'] = compute_pi0("../../results/K562-Exp-Lib-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.parquet")

pi0s[names['exp']][condition['refined']]['1'] = compute_pi0("../../results/K562-Exp-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.parquet")
pi0s[names['exp']][condition['refined']]['2'] = compute_pi0("../../results/K562-Exp-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.parquet")
pi0s[names['exp']][condition['refined']]['3'] = compute_pi0("../../results/K562-Exp-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.parquet")

pi0s[names['exp']][condition['tl']]['1'] = compute_pi0_oswpq("../../results/K562-Exp-Lib-Refined-GPF-Analysis/osw_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.oswpq")
pi0s[names['exp']][condition['tl']]['2'] = compute_pi0_oswpq("../../results/K562-Exp-Lib-Refined-GPF-Analysis/osw_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.oswpq")
pi0s[names['exp']][condition['tl']]['3'] = compute_pi0_oswpq("../../results/K562-Exp-Lib-Refined-GPF-Analysis/osw_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.oswpq")

pi0s[names['exp']][condition['onlyFilter']]['1'] = compute_pi0("../../results/K562-Exp-Lib-Refined-GPF-Analysis/osw_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.parquet")
pi0s[names['exp']][condition['onlyFilter']]['2'] = compute_pi0("../../results/K562-Exp-Lib-Refined-GPF-Analysis/osw_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.parquet")
pi0s[names['exp']][condition['onlyFilter']]['3'] = compute_pi0("../../results/K562-Exp-Lib-Refined-GPF-Analysis/osw_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.parquet")


# Silico data
pi0s[names['silico']][condition['orig']]['1'] = compute_pi0_oswpq("../../results/K562-PeptDeep-NoMods-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.oswpq")
pi0s[names['silico']][condition['orig']]['2'] = compute_pi0_oswpq("../../results/K562-PeptDeep-NoMods-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.oswpq")
pi0s[names['silico']][condition['orig']]['3'] = compute_pi0_oswpq("../../results/K562-PeptDeep-NoMods-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.oswpq")

pi0s[names['silico']][condition['refined']]['1'] = compute_pi0("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.parquet")
pi0s[names['silico']][condition['refined']]['2'] = compute_pi0("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.parquet")
pi0s[names['silico']][condition['refined']]['3'] = compute_pi0("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.parquet")

pi0s[names['silico']][condition['tl'] ]['1'] = compute_pi0_oswpq("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/osw_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.oswpq")
pi0s[names['silico']][condition['tl'] ]['2'] = compute_pi0_oswpq("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/osw_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.oswpq")
pi0s[names['silico']][condition['tl'] ]['3'] = compute_pi0_oswpq("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/osw_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.oswpq")

pi0s[names['silico']][condition['onlyFilter']]['1'] = compute_pi0("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/osw_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.parquet")
pi0s[names['silico']][condition['onlyFilter']]['2'] = compute_pi0("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/osw_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.parquet")
pi0s[names['silico']][condition['onlyFilter']]['3'] = compute_pi0("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/osw_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.parquet")

../../results/K562-Bruker-Library-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.parquet
../../results/K562-Bruker-Library-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.parquet
../../results/K562-Bruker-Library-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.parquet
../../results/K562-Bruker-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.parquet
../../results/K562-Bruker-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Sl

In [8]:
avg_pi0est = defaultdict(dict)
for lib, v in pi0s.items():
    for cond, vv in v.items():
        avg_pi0est[lib][cond] = np.mean([i[0] for i in vv.values()])

avg_pi0est = pd.DataFrame(avg_pi0est).reset_index(names='Condition').melt(id_vars='Condition', var_name='Library', value_name='pi0')

In [9]:
avg_jaccards = defaultdict(dict)
for lib, v in osw.items():
    for cond, vv in v.items():
        avg_jaccards[lib][cond] = avg_jaccard_index(vv)

avg_jaccards_osw = pd.DataFrame(avg_jaccards).reset_index(names='Condition').melt(id_vars='Condition', var_name='Library', value_name='Jaccard Index')

In [10]:
avg_jaccards = defaultdict(dict)
for lib, v in diann.items():
    for cond, vv in v.items():
        avg_jaccards[lib][cond] = avg_jaccard_index(vv)

avg_jaccards_diann = pd.DataFrame(avg_jaccards).reset_index(names='Condition').melt(id_vars='Condition', var_name='Library', value_name='Jaccard Index')

In [11]:
avg_ids = defaultdict(dict)
for lib, v in osw.items():
    for cond, vv in v.items():
        avg_ids[lib][cond] = np.array([ len(i) for i in vv.values() ] ).mean()

avg_ids_osw = pd.DataFrame(avg_ids).reset_index(names='Condition').melt(id_vars='Condition', var_name='Library', value_name='Average # IDs')

avg_ids = defaultdict(dict)
for lib, v in diann.items():
    for cond, vv in v.items():
        avg_ids[lib][cond] = np.array([ len(i) for i in vv.values() ] ).mean()

avg_ids_diann = pd.DataFrame(avg_ids).reset_index(names='Condition').melt(id_vars='Condition', var_name='Library', value_name='Average # IDs')

In [12]:
avg_jaccards_diann = pd.merge(avg_jaccards_diann, avg_pi0est).merge(avg_ids_diann)
avg_jaccards_osw = pd.merge(avg_jaccards_osw, avg_pi0est).merge(avg_ids_osw)

## **Load DataFrames**

In [13]:
osw_df = { n:defaultdict(dict) for n in names.values() }

# Bruker data
osw_df[names['bruker']][condition['orig']]['1'] = getPrecursorDf_Characteristics("../../results/K562-Bruker-Library-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.parquet")
osw_df[names['bruker']][condition['orig']]['2'] = getPrecursorDf_Characteristics("../../results/K562-Bruker-Library-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.parquet")
osw_df[names['bruker']][condition['orig']]['3'] = getPrecursorDf_Characteristics("../../results/K562-Bruker-Library-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.parquet")

osw_df[names['bruker']][condition['refined']]['1'] = getPrecursorDf_Characteristics("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.parquet")
osw_df[names['bruker']][condition['refined']]['2'] = getPrecursorDf_Characteristics("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.parquet")
osw_df[names['bruker']][condition['refined']]['3'] = getPrecursorDf_Characteristics("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.parquet")

osw_df[names['bruker']][condition['tl']]['1'] = getPrecursorDf_Characteristics_oswpq("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/osw_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.oswpq")
osw_df[names['bruker']][condition['tl']]['2'] = getPrecursorDf_Characteristics_oswpq("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/osw_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.oswpq")
osw_df[names['bruker']][condition['tl']]['3'] = getPrecursorDf_Characteristics_oswpq("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/osw_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.oswpq")

osw_df[names['bruker']][condition['onlyFilter']]['1'] = getPrecursorDf_Characteristics("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/osw_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.parquet")
osw_df[names['bruker']][condition['onlyFilter']]['2'] = getPrecursorDf_Characteristics("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/osw_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.parquet")
osw_df[names['bruker']][condition['onlyFilter']]['3'] = getPrecursorDf_Characteristics("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/osw_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.parquet")

# Panhuman data
osw_df[names['panhuman']][condition['orig']]['1'] = getPrecursorDf_Characteristics_oswpq("../../results/K562-PanHuman-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_SVM/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.oswpq/")
osw_df[names['panhuman']][condition['orig']]['2'] = getPrecursorDf_Characteristics_oswpq("../../results/K562-PanHuman-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_SVM/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.oswpq/")
osw_df[names['panhuman']][condition['orig']]['3'] = getPrecursorDf_Characteristics_oswpq("../../results/K562-PanHuman-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_SVM/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.oswpq/")

osw_df[names['panhuman']][condition['refined']]['1'] = getPrecursorDf_Characteristics_oswpq("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_SVM/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.oswpq/")
osw_df[names['panhuman']][condition['refined']]['2'] = getPrecursorDf_Characteristics_oswpq("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_SVM/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.oswpq/")
osw_df[names['panhuman']][condition['refined']]['3'] = getPrecursorDf_Characteristics_oswpq("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_SVM/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.oswpq/")

osw_df[names['panhuman']][condition['tl'] ]['1'] = getPrecursorDf_Characteristics_oswpq("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/osw_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_SVM/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.oswpq/")
osw_df[names['panhuman']][condition['tl'] ]['2'] = getPrecursorDf_Characteristics_oswpq("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/osw_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_SVM/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.oswpq/")
osw_df[names['panhuman']][condition['tl'] ]['3'] = getPrecursorDf_Characteristics_oswpq("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/osw_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_SVM/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.oswpq/")

osw_df[names['panhuman']][condition['onlyFilter'] ]['1'] = getPrecursorDf_Characteristics_oswpq("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/osw_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_SVM/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.oswpq/")
osw_df[names['panhuman']][condition['onlyFilter'] ]['2'] = getPrecursorDf_Characteristics_oswpq("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/osw_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_SVM/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.oswpq/")
osw_df[names['panhuman']][condition['onlyFilter'] ]['3'] = getPrecursorDf_Characteristics_oswpq("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/osw_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_SVM/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.oswpq/")

# Experimental data
osw_df[names['exp']][condition['orig']]['1'] = getPrecursorDf_Characteristics("../../results/K562-Exp-Lib-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.parquet")
osw_df[names['exp']][condition['orig']]['2'] = getPrecursorDf_Characteristics("../../results/K562-Exp-Lib-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.parquet")
osw_df[names['exp']][condition['orig']]['3'] = getPrecursorDf_Characteristics("../../results/K562-Exp-Lib-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.parquet")

osw_df[names['exp']][condition['refined']]['1'] = getPrecursorDf_Characteristics("../../results/K562-Exp-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.parquet")
osw_df[names['exp']][condition['refined']]['2'] = getPrecursorDf_Characteristics("../../results/K562-Exp-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.parquet")
osw_df[names['exp']][condition['refined']]['3'] = getPrecursorDf_Characteristics("../../results/K562-Exp-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.parquet")

osw_df[names['exp']][condition['tl']]['1'] = getPrecursorDf_Characteristics_oswpq("../../results/K562-Exp-Lib-Refined-GPF-Analysis/osw_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.oswpq")
osw_df[names['exp']][condition['tl']]['2'] = getPrecursorDf_Characteristics_oswpq("../../results/K562-Exp-Lib-Refined-GPF-Analysis/osw_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.oswpq")
osw_df[names['exp']][condition['tl']]['3'] = getPrecursorDf_Characteristics_oswpq("../../results/K562-Exp-Lib-Refined-GPF-Analysis/osw_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.oswpq")

osw_df[names['exp']][condition['onlyFilter']]['1'] = getPrecursorDf_Characteristics("../../results/K562-Exp-Lib-Refined-GPF-Analysis/osw_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.parquet")
osw_df[names['exp']][condition['onlyFilter']]['2'] = getPrecursorDf_Characteristics("../../results/K562-Exp-Lib-Refined-GPF-Analysis/osw_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.parquet")
osw_df[names['exp']][condition['onlyFilter']]['3'] = getPrecursorDf_Characteristics("../../results/K562-Exp-Lib-Refined-GPF-Analysis/osw_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.parquet")


# Silico data
osw_df[names['silico']][condition['orig']]['1'] = getPrecursorDf_Characteristics_oswpq("../../results/K562-PeptDeep-NoMods-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.oswpq")
osw_df[names['silico']][condition['orig']]['2'] = getPrecursorDf_Characteristics_oswpq("../../results/K562-PeptDeep-NoMods-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.oswpq")
osw_df[names['silico']][condition['orig']]['3'] = getPrecursorDf_Characteristics_oswpq("../../results/K562-PeptDeep-NoMods-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.oswpq")

osw_df[names['silico']][condition['refined']]['1'] = getPrecursorDf_Characteristics("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.parquet")
osw_df[names['silico']][condition['refined']]['2'] = getPrecursorDf_Characteristics("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.parquet")
osw_df[names['silico']][condition['refined']]['3'] = getPrecursorDf_Characteristics("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.parquet")

osw_df[names['silico']][condition['tl'] ]['1'] = getPrecursorDf_Characteristics_oswpq("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/osw_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.oswpq")
osw_df[names['silico']][condition['tl'] ]['2'] = getPrecursorDf_Characteristics_oswpq("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/osw_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.oswpq")
osw_df[names['silico']][condition['tl'] ]['3'] = getPrecursorDf_Characteristics_oswpq("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/osw_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.oswpq")

osw_df[names['silico']][condition['onlyFilter']]['1'] = getPrecursorDf_Characteristics("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/osw_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.parquet")
osw_df[names['silico']][condition['onlyFilter']]['2'] = getPrecursorDf_Characteristics("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/osw_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.parquet")
osw_df[names['silico']][condition['onlyFilter']]['3'] = getPrecursorDf_Characteristics("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/osw_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.parquet")

../../results/K562-Bruker-Library-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.parquet
../../results/K562-Bruker-Library-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021.parquet
../../results/K562-Bruker-Library-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021.parquet
../../results/K562-Bruker-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.parquet
../../results/K562-Bruker-Lib-Refined-GPF-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Sl

In [14]:
diann_df = { n:defaultdict(dict) for n in names.values() }

# Bruker data
diann_df[names['bruker']][condition['orig']]['1'] = getPrecursorDf_Characteristics_Diann("../../results/K562-Bruker-Library-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv")
diann_df[names['bruker']][condition['orig']]['2'] = getPrecursorDf_Characteristics_Diann("../../results/K562-Bruker-Library-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/report.tsv")
diann_df[names['bruker']][condition['orig']]['3'] = getPrecursorDf_Characteristics_Diann("../../results/K562-Bruker-Library-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/report.tsv")

diann_df[names['bruker']][condition['refined']]['1'] = getPrecursorDf_Characteristics_Diann("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv")
diann_df[names['bruker']][condition['refined']]['2'] = getPrecursorDf_Characteristics_Diann("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/report.tsv")
diann_df[names['bruker']][condition['refined']]['3'] = getPrecursorDf_Characteristics_Diann("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/report.tsv")

diann_df[names['bruker']][condition['tl']]['1'] = getPrecursorDf_Characteristics_Diann("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/diann_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv")
diann_df[names['bruker']][condition['tl']]['2'] = getPrecursorDf_Characteristics_Diann("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/diann_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/report.tsv")
diann_df[names['bruker']][condition['tl']]['3'] = getPrecursorDf_Characteristics_Diann("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/diann_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/report.tsv")

diann_df[names['bruker']][condition['onlyFilter']]['1'] = getPrecursorDf_Characteristics_Diann("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/diann_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv")
diann_df[names['bruker']][condition['onlyFilter']]['2'] = getPrecursorDf_Characteristics_Diann("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/diann_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/report.tsv")
diann_df[names['bruker']][condition['onlyFilter']]['3'] = getPrecursorDf_Characteristics_Diann("../../results/K562-Bruker-Lib-Refined-GPF-Analysis/diann_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/report.tsv")


# Silico data
diann_df[names['silico']][condition['orig']]['1'] = getPrecursorDf_Characteristics_Diann("../../results/K562-PeptDeep-NoMods-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv")
diann_df[names['silico']][condition['orig']]['2'] = getPrecursorDf_Characteristics_Diann("../../results/K562-PeptDeep-NoMods-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/report.tsv")
diann_df[names['silico']][condition['orig']]['3'] = getPrecursorDf_Characteristics_Diann("../../results/K562-PeptDeep-NoMods-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/report.tsv")

diann_df[names['silico']][condition['refined']]['1'] = getPrecursorDf_Characteristics_Diann("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv")
diann_df[names['silico']][condition['refined']]['2'] = getPrecursorDf_Characteristics_Diann("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/report.tsv")
diann_df[names['silico']][condition['refined']]['3'] = getPrecursorDf_Characteristics_Diann("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/report.tsv")

diann_df[names['silico']][condition['tl']]['1'] = getPrecursorDf_Characteristics_Diann("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/diann_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv")
diann_df[names['silico']][condition['tl']]['2'] = getPrecursorDf_Characteristics_Diann("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/diann_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/report.tsv")
diann_df[names['silico']][condition['tl']]['3'] = getPrecursorDf_Characteristics_Diann("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/diann_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/report.tsv")

diann_df[names['silico']][condition['onlyFilter']]['1'] = getPrecursorDf_Characteristics_Diann("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/diann_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv")
diann_df[names['silico']][condition['onlyFilter']]['2'] = getPrecursorDf_Characteristics_Diann("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/diann_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/report.tsv")
diann_df[names['silico']][condition['onlyFilter']]['3'] = getPrecursorDf_Characteristics_Diann("../../results/K562-PeptDeep-NoMods-Lib-Refined-GPF-Analysis/diann_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/report.tsv")


# exp data
diann_df[names['exp']][condition['orig']]['1'] = getPrecursorDf_Characteristics_Diann("../../results/K562-Exp-Lib-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv")
diann_df[names['exp']][condition['orig']]['2'] = getPrecursorDf_Characteristics_Diann("../../results/K562-Exp-Lib-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/report.tsv")
diann_df[names['exp']][condition['orig']]['3'] = getPrecursorDf_Characteristics_Diann("../../results/K562-Exp-Lib-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/report.tsv")

diann_df[names['exp']][condition['refined']]['1'] = getPrecursorDf_Characteristics_Diann("../../results/K562-Exp-Lib-Refined-GPF-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv", infer_schema_length=1000)
diann_df[names['exp']][condition['refined']]['2'] = getPrecursorDf_Characteristics_Diann("../../results/K562-Exp-Lib-Refined-GPF-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/report.tsv", infer_schema_length=1000)
diann_df[names['exp']][condition['refined']]['3'] = getPrecursorDf_Characteristics_Diann("../../results/K562-Exp-Lib-Refined-GPF-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/report.tsv", infer_schema_length=1000)

diann_df[names['exp']][condition['tl']]['1'] = getPrecursorDf_Characteristics_Diann("../../results/K562-Exp-Lib-Refined-GPF-Analysis/diann_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv", infer_schema_length=1000)
diann_df[names['exp']][condition['tl']]['2'] = getPrecursorDf_Characteristics_Diann("../../results/K562-Exp-Lib-Refined-GPF-Analysis/diann_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/report.tsv", infer_schema_length=1000)
diann_df[names['exp']][condition['tl']]['3'] = getPrecursorDf_Characteristics_Diann("../../results/K562-Exp-Lib-Refined-GPF-Analysis/diann_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/report.tsv", infer_schema_length=1000)

diann_df[names['exp']][condition['onlyFilter']]['1'] = getPrecursorDf_Characteristics_Diann("../../results/K562-Exp-Lib-Refined-GPF-Analysis/diann_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv", infer_schema_length=1000)
diann_df[names['exp']][condition['onlyFilter']]['2'] = getPrecursorDf_Characteristics_Diann("../../results/K562-Exp-Lib-Refined-GPF-Analysis/diann_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/report.tsv", infer_schema_length=1000)
diann_df[names['exp']][condition['onlyFilter']]['3'] = getPrecursorDf_Characteristics_Diann("../../results/K562-Exp-Lib-Refined-GPF-Analysis/diann_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/report.tsv", infer_schema_length=1000)


# Panhuman data
diann_df[names['panhuman']][condition['orig']]['1'] = getPrecursorDf_Characteristics_Diann("../../results/K562-PanHuman-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv")
diann_df[names['panhuman']][condition['orig']]['2'] = getPrecursorDf_Characteristics_Diann("../../results/K562-PanHuman-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/report.tsv")
diann_df[names['panhuman']][condition['orig']]['3'] = getPrecursorDf_Characteristics_Diann("../../results/K562-PanHuman-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/report.tsv")

diann_df[names['panhuman']][condition['refined']]['1'] = getPrecursorDf_Characteristics_Diann("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv")
diann_df[names['panhuman']][condition['refined']]['2'] = getPrecursorDf_Characteristics_Diann("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/report.tsv")
diann_df[names['panhuman']][condition['refined']]['3'] = getPrecursorDf_Characteristics_Diann("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/report.tsv")

diann_df[names['panhuman']][condition['tl']]['1'] = getPrecursorDf_Characteristics_Diann("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/diann_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv")
diann_df[names['panhuman']][condition['tl']]['2'] = getPrecursorDf_Characteristics_Diann("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/diann_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/report.tsv")
diann_df[names['panhuman']][condition['tl']]['3'] = getPrecursorDf_Characteristics_Diann("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/diann_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/report.tsv")

diann_df[names['panhuman']][condition['onlyFilter']]['1'] = getPrecursorDf_Characteristics_Diann("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/diann_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv")
diann_df[names['panhuman']][condition['onlyFilter']]['2'] = getPrecursorDf_Characteristics_Diann("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/diann_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/report.tsv")
diann_df[names['panhuman']][condition['onlyFilter']]['3'] = getPrecursorDf_Characteristics_Diann("../../results/K562-PanHuman-Lib-Refined-GPF-Analysis/diann_onlyFilter/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/report.tsv")

../../results/K562-Bruker-Library-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv
../../results/K562-Bruker-Library-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/report.tsv
../../results/K562-Bruker-Library-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/report.tsv
../../results/K562-Bruker-Lib-Refined-GPF-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv
../../results/K562-Bruker-Lib-Refined-GPF-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_2_Slot1-5_1_1331_6-28-2021/report.tsv
../../results/K562-Bruker-Lib-Refined-GPF-Analysis/diann/Rost_DIApy3_SP2um_90min_250ngK562_100nL_3_Slot1-5_1_1332_6-28-2021/report.tsv
../../results/K562-Bruker-Lib-Refined-GPF-Analysis/diann_tl/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/report.tsv
../../results/K562-Bruker-Lib-Refined-GPF-Analysis/diann_tl/Rost_DIApy3_SP2u

## **Compute RT**

In [15]:
rt_stds = defaultdict(dict)
for lib, v in osw_df.items():
    for cond, vv in v.items():
        tmp = []
        for rep, vvv in vv.items():
            tmp.append(vvv['FEATURE.DELTA_RT'].std()) 
        
        rt_stds[lib][cond] = np.array(tmp).mean()

rt_stds_osw = pd.DataFrame(rt_stds).reset_index(names='Condition').melt(id_vars='Condition', var_name='Library', value_name='Retention Time Std.')


rt_stds = defaultdict(dict)
for lib, v in diann_df.items():
    for cond, vv in v.items():
        tmp = []
        for rep, vvv in vv.items():
            tmp.append(vvv['deltaRT'].std()) 
        
        rt_stds[lib][cond] = np.array(tmp).mean()

rt_stds_diann = pd.DataFrame(rt_stds).reset_index(names='Condition').melt(id_vars='Condition', var_name='Library', value_name='Retention Time Std.')
rt_stds_diann['Software'] = 'DIA-NN'


#### **Put Everything Together**

In [16]:
metrics_osw = pd.merge(avg_jaccards_osw, rt_stds_osw) 
metrics_osw['Software'] = 'OpenSWATH'
metrics_diann = pd.merge(avg_jaccards_diann, rt_stds_diann)
metrics_diann['Software'] = 'DIA-NN'

metrics = pd.concat([metrics_diann, metrics_osw])

In [17]:
metrics['-log(pi0)'] = -np.log(metrics['pi0'])

In [18]:
def plot(df, fontsize_title=13, fontsize_large=11, fontsize_medium=9):
    """Plot Jaccard Index vs. -log(pi0) and Retention Time Std for DIA-NN and OpenSWATH."""
    
    # Compute -log10(pi0) if not already present
    if '-log(pi0)' not in df.columns:
        df['-log(pi0)'] = -np.log(df['pi0'])

    fig, axes = plt.subplots(2, 2, figsize=(9, 7), sharey=True)
    software_list = ['DIA-NN', 'OpenSWATH']
    x_metrics = ['-log(pi0)', 'Retention Time Std.']
    x_labels = ["-log(π₀)", "Retention Time Std."]

    def annotate_r2(ax, x, y, data, pos_x=0.05, pos_y=0.95):
        """Compute and annotate R²."""
        valid = data[[x, y]].dropna()
        if len(valid) > 1:
            r, _ = pearsonr(valid[x], valid[y])
            r2 = r**2
            ax.text(
                pos_x, pos_y,
                f"$R^2$ = {r2:.2f}",
                transform=ax.transAxes,
                fontsize=11,
                verticalalignment='top',
                bbox=dict(boxstyle="round,pad=0.3", facecolor='white', alpha=0.7)
            )

    for i, software in enumerate(software_list):
        df_sub = df[df['Software'] == software]
        for j, xvar in enumerate(x_metrics):
            ax = axes[i, j]
            sns.scatterplot(
                data=df_sub,
                x=xvar,
                y='Jaccard Index',
                s=80,
                color='tab:blue',
                ax=ax,
            )

            
            ax.tick_params(axis='both', labelsize=fontsize_large)

            ax.set_xlabel(xvar)
            if j == 0:
                ax.set_ylabel('Jaccard Index', fontsize=fontsize_large)
                pos_x = 0.05
                ax.set_xlim(-0.05, 4)


            else:
                ax.set_ylabel('')
                pos_x = 0.7
                ax.set_xlim(0, 200)

            if i == 1:
                ax.set_xlabel(x_labels[j], fontsize=fontsize_large)
            else:
                ax.set_xlabel('')

            ax.grid()

            annotate_r2(ax, xvar, 'Jaccard Index', df_sub, pos_x=pos_x)


    # Add centered row labels
    fig.text(0.5, 0.975, 'DIA-NN', ha='center', va='top', fontsize=fontsize_title)
    fig.text(0.5, 0.515, 'OpenSWATH', ha='center', va='top', fontsize=fontsize_title)

    # Adjust layout to add space between rows
    plt.subplots_adjust(hspace=0.30, top=0.93, bottom=0.08)
    fig.savefig('Figure-S9.png', dpi=400, bbox_inches='tight')



plot(metrics)
plt.show()
